# Milestone 2

This notebook implements an end-to-end real-time streaming pipeline for a food delivery simulation:

- **Two AVRO producers** push events to Azure Event Hubs (`group_01_orders` and `group_01_courier_state_events`).
- **Spark Structured Streaming** reads, deserializes, and analyses both streams in real time.
- **Analytical queries** cover order lifecycle metrics, zone demand, cancellation rates, and courier activity.
- **Parquet files** are written to Azure Blob Storage for downstream consumption.

**Event Hub Namespace:** `iesstsabbadbab-grp-01-05`  
**Event Hubs:** `group_01_orders`, `group_01_courier_state_events`  
**GitHub:** https://github.com/jeancarlosie/group-1

## 0. Configuration

In [1]:
# Azure Event Hub
event_hub_namespace = 'iesstsabbadbab-grp-01-05'

# Orders Event Hub
orders_eventhub_name     = 'group_01_orders'
orders_producer_conn_str = 'Endpoint=sb://iesstsabbadbab-grp-01-05.servicebus.windows.net/;SharedAccessKeyName=Producer;SharedAccessKey=IG3+QBLn2NG18cvjKM+Rv487HZkUQWpm2+AEhBFU9Ak=;EntityPath=group_01_orders'
orders_consumer_conn_str = 'Endpoint=sb://iesstsabbadbab-grp-01-05.servicebus.windows.net/;SharedAccessKeyName=Consumer;SharedAccessKey=Rex4xptVTzkHMrWp17loxo6RWM8h76qWc+AEhO4uEWU=;EntityPath=group_01_orders'

# Courier State Events Hub
courier_eventhub_name     = 'group_01_courier_state_events'
courier_producer_conn_str = 'Endpoint=sb://iesstsabbadbab-grp-01-05.servicebus.windows.net/;SharedAccessKeyName=Producer;SharedAccessKey=l56xiUtGweKvmFEC2qh6eSCTm3MCzeiig+AEhL9JTAk=;EntityPath=group_01_courier_state_events'
courier_consumer_conn_str = 'Endpoint=sb://iesstsabbadbab-grp-01-05.servicebus.windows.net/;SharedAccessKeyName=Consumer;SharedAccessKey=ahLAAZ4MaTdsp0lojerR+wbCcwIl7JN1d+AEhBHL7Zk=;EntityPath=group_01_courier_state_events'
# Azure Blob Storage
account_name   = 'iesstsabbadbab'
account_key    = 'QOlT3Y2sp4wDkdW+Yb3Cddfo9PdoG8+ORBZy0ZDMZCiQtkK+rtI+Ri1WVXopfWgyoqC8N69czF9L+AStbdQ8bA=='
container_name = 'group01output'

print('Configuration loaded.')
print(f'Namespace: {event_hub_namespace}')
print(f'Orders hub: {orders_eventhub_name}')
print(f'Courier hub: {courier_eventhub_name}')
print(f'Blob: {account_name}/{container_name}')

Configuration loaded.
Namespace: iesstsabbadbab-grp-01-05
Orders hub: group_01_orders
Courier hub: group_01_courier_state_events
Blob: iesstsabbadbab/group01output


## 1. Install Dependencies

In [2]:
!pip install -q fastavro confluent-kafka

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 91.7 MB/s eta 0:00:00


In [3]:
!pip install -q azure-storage-blob

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 kB 6.7 MB/s eta 0:00:00


## 2. AVRO Schemas

### 2.1 group_01_orders

In [4]:
order_schema_str = """
{
  "type": "record",
  "name": "OrderEvent",
  "namespace": "com.fooddelivery.events",
  "doc": "Represents lifecycle events of a food delivery order",
  "fields": [
    {"name": "schema_version", "type": "int",    "default": 1},
    {"name": "event_id",       "type": "string"},
    {"name": "event_sequence", "type": ["null", "int"],    "default": null},
    {"name": "event_type",
     "type": {
       "type": "enum",
       "name": "OrderEventType",
       "symbols": [
         "ORDER_CREATED", "RESTAURANT_ACCEPTED", "PREP_STARTED",
         "PREP_DONE", "COURIER_ASSIGNED", "PICKED_UP", "DELIVERED", "CANCELLED"
       ]
     }
    },
    {"name": "order_id",      "type": "string"},
    {"name": "restaurant_id", "type": "string"},
    {"name": "zone_id",       "type": "string"},
    {"name": "courier_id",    "type": ["null", "string"], "default": null},
    {"name": "order_value",   "type": ["null", "double"], "default": null},
    {"name": "event_time",    "type": "long"},
    {"name": "ingest_time",   "type": "long"}
  ]
}
"""

print("Order schema defined.")

Order schema defined.


### 2.2 group_01_courier_state_events

In [5]:
courier_schema_str = """
{
  "type": "record",
  "name": "CourierStateEvent",
  "namespace": "com.fooddelivery.events",
  "doc": "Represents real-time courier status and availability events",
  "fields": [
    {"name": "schema_version", "type": "int",    "default": 1},
    {"name": "event_id",       "type": "string"},
    {"name": "event_type",
     "type": {
       "type": "enum",
       "name": "CourierEventType",
       "symbols": [
         "ONLINE", "OFFLINE", "LOCATION_UPDATE",
         "ARRIVED_RESTAURANT", "PICKED_UP_ORDER", "ARRIVED_CUSTOMER"
       ]
     }
    },
    {"name": "courier_id", "type": "string"},
    {"name": "order_id",   "type": ["null", "string"], "default": null},
    {"name": "zone_id",    "type": "string"},
    {"name": "latitude",   "type": ["null", "double"], "default": null},
    {"name": "longitude",  "type": ["null", "double"], "default": null},
    {"name": "event_time",  "type": "long"},
    {"name": "ingest_time", "type": "long"}
  ]
}
"""

print("Courier schema defined.")

Courier schema defined.


## 3. Python Producers

- Script: `avro_orders_producer.py` | Event Hub: `group_01_orders` | Event Types: ORDER_CREATED → … → DELIVERED / CANCELLED
- Script: `avro_courier_producer.py` | Event Hub: `group_01_courier_state_events` | Event Types: ONLINE, OFFLINE, LOCATION_UPDATE, …

### 3.1 avro_orders_producer.py

In [6]:
%%writefile avro_orders_producer.py

import sys, uuid, random, io, time
from datetime import datetime, timedelta
import fastavro
from fastavro import parse_schema
from confluent_kafka import Producer

# AVRO Schema
order_schema_dict = {
    "type": "record",
    "name": "OrderEvent",
    "namespace": "com.fooddelivery.events",
    "fields": [
        {"name": "schema_version", "type": "int",  "default": 1},
        {"name": "event_id",       "type": "string"},
        {"name": "event_sequence", "type": ["null", "int"], "default": None},
        {"name": "event_type",
         "type": {
             "type": "enum", "name": "OrderEventType",
             "symbols": ["ORDER_CREATED", "RESTAURANT_ACCEPTED", "PREP_STARTED",
                         "PREP_DONE", "COURIER_ASSIGNED", "PICKED_UP", "DELIVERED", "CANCELLED"]
         }},
        {"name": "order_id",      "type": "string"},
        {"name": "restaurant_id", "type": "string"},
        {"name": "zone_id",       "type": "string"},
        {"name": "courier_id",    "type": ["null", "string"], "default": None},
        {"name": "order_value",   "type": ["null", "double"], "default": None},
        {"name": "event_time",  "type": "long"},
        {"name": "ingest_time", "type": "long"}
    ]
}
parsed_schema = parse_schema(order_schema_dict)

# Simulation parameters (config.yaml)
ZONES = [f'zone_{i}' for i in range(5)]
RESTAURANTS = {z: [f'rest_{z}_{i}' for i in range(20)] for z in ZONES}
COURIERS = [f'courier_{i}' for i in range(50)]
ZONE_WEIGHTS = [1.0, 1.0, 1.8, 0.7, 1.3]

CANCELLATION_PROB = 0.10
DUPLICATE_PROB = 0.05
LATE_EVENT_PROB = 0.10
MISSING_STEP_PROB = 0.05
ANOMALY_PROB = 0.05
BASE_ORDERS_PER_BATCH = 5
BATCH_SLEEP_SECONDS = 1.0

# Helpers
def millis(dt):
    return int(dt.timestamp() * 1000)

def maybe_late(t):
    if random.random() < LATE_EVENT_PROB:
        return t + timedelta(minutes=random.randint(2, 10))
    return t

def avro_serialize(record):
    with io.BytesIO() as buf:
        fastavro.schemaless_writer(buf, parsed_schema, record)
        return buf.getvalue()

def delivery_report(err, msg):
    if err:
        print(f'[ERROR] Delivery failed: {err}')
    else:
        print(f'[OK] {msg.topic()} partition={msg.partition()} offset={msg.offset()}')

# Argument parsing
if len(sys.argv) < 4:
    print('Usage: python avro_orders_producer.py <namespace> <eventhub_name> <connection_string>')
    sys.exit(1)

event_hub_namespace = sys.argv[1]
eventhub_name       = sys.argv[2]
connection_string   = sys.argv[3]

# Kafka / Event Hub Producer config
conf = {
    'bootstrap.servers': f'{event_hub_namespace}.servicebus.windows.net:9093',
    'security.protocol': 'SASL_SSL',
    'sasl.mechanisms':   'PLAIN',
    'sasl.username':     '$ConnectionString',
    'sasl.password':     connection_string,
    'client.id':         'group01-orders-producer'
}
producer = Producer(**conf)
print(f'Orders producer started -> {eventhub_name} @ {event_hub_namespace}')

# Main production loop
batch = 0
while True:
    batch += 1
    now = datetime.utcnow()

    hour = now.hour
    multiplier = 1.0
    if 11 <= hour <= 14:
        multiplier = 2.0
    elif 18 <= hour <= 21:
        multiplier = 2.5
    if now.weekday() >= 5:
        multiplier *= 1.3

    n_orders = max(1, int(BASE_ORDERS_PER_BATCH * multiplier))

    for _ in range(n_orders):
        order_id    = str(uuid.uuid4())
        zone_id     = random.choices(ZONES, weights=ZONE_WEIGHTS, k=1)[0]
        restaurant  = random.choice(RESTAURANTS[zone_id])
        courier_id  = random.choice(COURIERS)
        order_value = round(min(max(random.lognormvariate(3, 0.5), 8), 80), 2)

        prep_min     = random.randint(5, 20)
        delivery_min = random.randint(10, 30)

        if random.random() < ANOMALY_PROB:
            r = random.random()
            if r < 0.2:
                delivery_min = -random.randint(1, 10)
            elif r < 0.6:
                delivery_min = random.randint(1, 5)
            else:
                delivery_min = random.randint(60, 120)

        t_created   = now
        t_accepted  = t_created  + timedelta(minutes=1)
        t_prep_done = t_accepted + timedelta(minutes=prep_min)
        t_pickup    = t_prep_done + timedelta(minutes=random.randint(1, 8))
        t_delivered = t_pickup + timedelta(minutes=delivery_min)

        skip_pickup = random.random() < MISSING_STEP_PROB
        cancelled   = random.random() < CANCELLATION_PROB

        def make_event(etype, t, cid=None, seq=None):
            return {
                'schema_version': 1,
                'event_id':       str(uuid.uuid4()),
                'event_sequence': seq,
                'event_type':     etype,
                'order_id':       order_id,
                'restaurant_id':  restaurant,
                'zone_id':        zone_id,
                'courier_id':     cid,
                'order_value':    order_value,
                'event_time':     millis(t),
                'ingest_time':    millis(maybe_late(t))
            }

        seq = 1
        events = []
        events.append(make_event('ORDER_CREATED',       t_created,  seq=seq)); seq += 1
        events.append(make_event('RESTAURANT_ACCEPTED', t_accepted, seq=seq)); seq += 1

        if random.random() < 0.95:
            events.append(make_event('PREP_STARTED',
                          t_accepted + timedelta(minutes=1), seq=seq)); seq += 1

        if cancelled:
            events.append(make_event('CANCELLED',
                          t_accepted + timedelta(minutes=random.randint(0, 3)), seq=seq))
        else:
            events.append(make_event('PREP_DONE',        t_prep_done, seq=seq)); seq += 1
            events.append(make_event('COURIER_ASSIGNED', t_prep_done, courier_id, seq=seq)); seq += 1
            if not skip_pickup:
                events.append(make_event('PICKED_UP', t_pickup, courier_id, seq=seq)); seq += 1
            events.append(make_event('DELIVERED', t_delivered, courier_id, seq=seq))

        for ev in events:
            copies = [ev]
            if random.random() < DUPLICATE_PROB:
                copies.append(ev.copy())
            for copy in copies:
                producer.produce(topic=eventhub_name, value=avro_serialize(copy), callback=delivery_report)
                producer.poll(0)

    producer.flush()
    print(f'[Batch {batch}] {n_orders} orders @ {now.strftime("%H:%M:%S")} (x{multiplier:.1f})')
    time.sleep(BATCH_SLEEP_SECONDS)

Writing avro_orders_producer.py


In [7]:
import subprocess, sys

orders_proc = subprocess.Popen(
    [sys.executable, 'avro_orders_producer.py',
     event_hub_namespace,
     orders_eventhub_name,
     orders_producer_conn_str],
    stdout=open('avro_orders_producer.log', 'w'),
    stderr=subprocess.STDOUT
)
print(f'Orders producer launched — PID {orders_proc.pid}')

Orders producer launched — PID 2916


### 3.2 avro_courier_producer.py

In [8]:
%%writefile avro_courier_producer.py

import sys, uuid, random, io, time
from datetime import datetime, timedelta
import fastavro
from fastavro import parse_schema
from confluent_kafka import Producer

# AVRO Schema
courier_schema_dict = {
    "type": "record",
    "name": "CourierStateEvent",
    "namespace": "com.fooddelivery.events",
    "fields": [
        {"name": "schema_version", "type": "int",  "default": 1},
        {"name": "event_id",   "type": "string"},
        {"name": "event_type",
         "type": {
             "type": "enum", "name": "CourierEventType",
             "symbols": ["ONLINE", "OFFLINE", "LOCATION_UPDATE",
                         "ARRIVED_RESTAURANT", "PICKED_UP_ORDER", "ARRIVED_CUSTOMER"]
         }},
        {"name": "courier_id", "type": "string"},
        {"name": "order_id",   "type": ["null", "string"], "default": None},
        {"name": "zone_id",    "type": "string"},
        {"name": "latitude",   "type": ["null", "double"], "default": None},
        {"name": "longitude",  "type": ["null", "double"], "default": None},
        {"name": "event_time",  "type": "long"},
        {"name": "ingest_time", "type": "long"}
    ]
}
parsed_schema = parse_schema(courier_schema_dict)

# Simulation parameters
ZONES    = [f'zone_{i}' for i in range(5)]
COURIERS = [f'courier_{i}' for i in range(50)]

DUPLICATE_PROB   = 0.05
LATE_EVENT_PROB  = 0.10
OFFLINE_PROB     = 0.05
BATCH_SLEEP_SECS = 0.5

ZONE_CENTERS = {f'zone_{i}': (40.0 + i * 0.02, -3.7 + i * 0.02) for i in range(5)}
courier_zone = {c: random.choice(ZONES) for c in COURIERS}

# Helpers
def millis(dt):
    return int(dt.timestamp() * 1000)

def maybe_late(t):
    if random.random() < LATE_EVENT_PROB:
        return t + timedelta(minutes=random.randint(2, 10))
    return t

def jitter(zone):
    lat, lon = ZONE_CENTERS[zone]
    return lat + random.uniform(-0.005, 0.005), lon + random.uniform(-0.005, 0.005)

def avro_serialize(record):
    with io.BytesIO() as buf:
        fastavro.schemaless_writer(buf, parsed_schema, record)
        return buf.getvalue()

def delivery_report(err, msg):
    if err:
        print(f'[ERROR] {err}')
    else:
        print(f'[OK] {msg.topic()} partition={msg.partition()} offset={msg.offset()}')

def make_courier_event(etype, cid, zone, t, order=None, lat=None, lon=None):
    return {
        'schema_version': 1,
        'event_id':   str(uuid.uuid4()),
        'event_type': etype,
        'courier_id': cid,
        'order_id':   order,
        'zone_id':    zone,
        'latitude':   lat,
        'longitude':  lon,
        'event_time':  millis(t),
        'ingest_time': millis(maybe_late(t))
    }

# Argument parsing
if len(sys.argv) < 4:
    print('Usage: python avro_courier_producer.py <namespace> <eventhub_name> <connection_string>')
    sys.exit(1)

event_hub_namespace = sys.argv[1]
eventhub_name       = sys.argv[2]
connection_string   = sys.argv[3]

# Kafka / Event Hub Producer config
conf = {
    'bootstrap.servers': f'{event_hub_namespace}.servicebus.windows.net:9093',
    'security.protocol': 'SASL_SSL',
    'sasl.mechanisms':   'PLAIN',
    'sasl.username':     '$ConnectionString',
    'sasl.password':     connection_string,
    'client.id':         'group01-courier-producer'
}
producer = Producer(**conf)
print(f'Courier producer started -> {eventhub_name} @ {event_hub_namespace}')

# Initial ONLINE burst for all 50 couriers
now = datetime.utcnow()
for cid in COURIERS:
    ev = make_courier_event('ONLINE', cid, courier_zone[cid], now)
    producer.produce(topic=eventhub_name, value=avro_serialize(ev), callback=delivery_report)
    producer.poll(0)
producer.flush()
print(f'Sent ONLINE event for {len(COURIERS)} couriers.')

# Main production loop
batch = 0
while True:
    batch += 1
    now = datetime.utcnow()
    events = []

    active = random.sample(COURIERS, k=random.randint(3, 10))

    for cid in active:
        zone     = courier_zone[cid]
        order_id = str(uuid.uuid4())
        lat, lon = jitter(zone)
        t = now

        sequence_type = random.choice(['pickup_delivery', 'restaurant_arrival', 'location_only'])

        if sequence_type == 'pickup_delivery':
            events.append(make_courier_event('ARRIVED_RESTAURANT', cid, zone, t, order_id))
            t += timedelta(seconds=random.randint(30, 120))
            events.append(make_courier_event('PICKED_UP_ORDER', cid, zone, t, order_id))
            t += timedelta(seconds=random.randint(60, 300))
            mid_lat, mid_lon = jitter(zone)
            events.append(make_courier_event('LOCATION_UPDATE', cid, zone, t, order_id, mid_lat, mid_lon))
            t += timedelta(seconds=random.randint(60, 300))
            events.append(make_courier_event('ARRIVED_CUSTOMER', cid, zone, t, order_id))
        elif sequence_type == 'restaurant_arrival':
            events.append(make_courier_event('ARRIVED_RESTAURANT', cid, zone, t, order_id))
        else:
            events.append(make_courier_event('LOCATION_UPDATE', cid, zone, t, None, lat, lon))

        if random.random() < OFFLINE_PROB:
            t_off = t + timedelta(minutes=1)
            t_on  = t_off + timedelta(minutes=random.randint(5, 20))
            events.append(make_courier_event('OFFLINE', cid, zone, t_off))
            events.append(make_courier_event('ONLINE',  cid, zone, t_on))

    for ev in events:
        copies = [ev]
        if random.random() < DUPLICATE_PROB:
            copies.append(ev.copy())
        for copy in copies:
            producer.produce(topic=eventhub_name, value=avro_serialize(copy), callback=delivery_report)
            producer.poll(0)

    producer.flush()
    print(f'[Batch {batch}] {len(events)} courier events @ {now.strftime("%H:%M:%S")}')
    time.sleep(BATCH_SLEEP_SECS)

Writing avro_courier_producer.py


In [9]:
import subprocess, sys

courier_proc = subprocess.Popen(
    [sys.executable, 'avro_courier_producer.py',
     event_hub_namespace,
     courier_eventhub_name,
     courier_producer_conn_str],
    stdout=open('avro_courier_producer.log', 'w'),
    stderr=subprocess.STDOUT
)
print(f'Courier producer launched — PID {courier_proc.pid}')

Courier producer launched — PID 2917


### 3.3 Test

In [10]:
import time

time.sleep(10)
print('Orders producer running:', orders_proc.poll() is None)
print('Courier producer running:', courier_proc.poll() is None)

print('\nOrders Producer (last 15 lines):')
with open('avro_orders_producer.log') as f:
    lines = f.readlines()
print(''.join(lines[-15:]))

print('Courier Producer (last 15 lines):')
with open('avro_courier_producer.log') as f:
    lines = f.readlines()
print(''.join(lines[-15:]))

Orders producer running: True
Courier producer running: True

Orders Producer (last 15 lines):
[OK] group_01_orders partition=2 offset=1853241
[OK] group_01_orders partition=2 offset=1853242
[OK] group_01_orders partition=2 offset=1853243
[OK] group_01_orders partition=2 offset=1853244
[OK] group_01_orders partition=2 offset=1853245
[OK] group_01_orders partition=2 offset=1853246
[OK] group_01_orders partition=2 offset=1853247
[OK] group_01_orders partition=2 offset=1853248
[OK] group_01_orders partition=2 offset=1853249
[OK] group_01_orders partition=2 offset=1853250
[Batch 4] 6 orders @ 16:39:43 (x1.3)
[OK] group_01_orders partition=2 offset=1853251
[OK] group_01_orders partition=2 offset=1853252
[OK] group_01_orders partition=2 offset=1853253
[OK] group_01_orders partition=2 offset=1853254
Courier Producer (last 15 lines):
[OK] group_01_courier_state_events partition=0 offset=628168
[OK] group_01_courier_state_events partition=0 offset=628169
[OK] group_01_courier_state_events parti

## 4. Spark Setup

In [11]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyspark==3.5.1'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'fastavro', 'confluent-kafka'], check=True)

import pyspark
print(f'PySpark installed: {pyspark.__version__}')

PySpark installed: 3.5.1


In [12]:
import os, subprocess

# Java 17 is the version installed in our Colab runtime
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ.pop('PYSPARK_SUBMIT_ARGS', None) # This clears any stale PySpark submit args from previous runs, in order to avoid conflicting JAR or config flags.

import pyspark
os.environ['SPARK_HOME'] = pyspark.__path__[0]

result = subprocess.run(['java', '-version'], capture_output=True, text=True)
print(result.stderr.split('\n')[0])
print(f'PySpark : {pyspark.__version__}')
print(f'JAVA_HOME: {os.environ["JAVA_HOME"]}')

openjdk version "17.0.18" 2026-01-20
PySpark : 3.5.1
JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64


In [13]:
# Environment ready
print('Environment check passed. Ready to create SparkSession.')

Environment check passed. Ready to create SparkSession.


## 5. Spark Session

Ten JAR packages are downloaded from Maven Central to `/content/jars/`,
and then passed to the SparkSession at startup:

- `spark-sql-kafka-0-10_2.12`: Kafka / Event Hubs Structured Streaming source
- `spark-avro_2.12`: enables `from_avro()` for AVRO deserialization
- `hadoop-azure` + `azure-storage`: WASB connector for reading/writing Parquet to Azure Blob Storage
- `kafka-clients`: low-level Kafka client required by the Spark Kafka connector
- `spark-token-provider-kafka-0-10_2.12`: handles authentication tokens for secured Kafka connections (required for SASL_SSL)
- `jetty-util` + `jetty-util-ajax`: HTTP utilities required by the Azure storage connector
- `wildfly-openssl`: native SSL provider used by the Kafka SASL_SSL handshake
- `commons-pool2`: connection pooling dependency for the Kafka connector

JARs are cached in `/content/jars/`. If a file already exists it is skipped,
so re-running this cell after a kernel restart only re-downloads what is missing.

In [14]:
import subprocess, os

jar_dir = "/content/jars"
os.makedirs(jar_dir, exist_ok=True)

jars_to_download = [
    ("org.apache.spark", "spark-sql-kafka-0-10_2.12", "3.5.1"),
    ("org.apache.spark", "spark-avro_2.12", "3.5.1"),
    ("org.apache.hadoop", "hadoop-azure", "3.3.4"),
    ("com.microsoft.azure", "azure-storage", "8.6.6"),
    ("org.apache.kafka", "kafka-clients", "3.4.1"),
    ("org.apache.spark", "spark-token-provider-kafka-0-10_2.12", "3.5.1"),
    ("org.eclipse.jetty", "jetty-util", "9.4.51.v20230217"),
    ("org.eclipse.jetty", "jetty-util-ajax", "9.4.51.v20230217"),
    ("org.wildfly.openssl", "wildfly-openssl", "1.0.7.Final"),
    ("org.apache.commons", "commons-pool2", "2.11.1"),
]

for group, artifact, version in jars_to_download:
    group_path = group.replace(".", "/")
    url = f"https://repo1.maven.org/maven2/{group_path}/{artifact}/{version}/{artifact}-{version}.jar"
    dest = f"{jar_dir}/{artifact}-{version}.jar"
    if not os.path.exists(dest):
        print(f"Downloading {artifact}-{version}...")
        r = subprocess.run(["wget", "-q", "-O", dest, url])
        if r.returncode != 0:
            print(f"  ⚠️ Failed: {url}")
        else:
            print(f"  ✅ OK")
    else:
        print(f"  Already exists: {artifact}")

jar_list = ",".join([f"{jar_dir}/{a}-{v}.jar" for _, a, v in jars_to_download])
print("\nAll JARs ready.")

  ✅ OK
  ✅ OK
  ✅ OK
  ✅ OK
  ✅ OK
  ✅ OK
  ✅ OK
  ✅ OK
  ✅ OK
  ✅ OK

All JARs ready.


In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.avro.functions import from_avro
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Group1_FoodDelivery_Streaming")
    .master("local[2]")
    .config("spark.jars", jar_list)
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.streaming.stopGracefullyOnShutdown", True)
    .config(f"fs.azure.account.key.{account_name}.blob.core.windows.net", account_key)
    .config("fs.azure", "org.apache.hadoop.fs.azure.NativeAzureFileSystem")
    .config("spark.executor.extraJavaOptions",
            "-Djava.security.auth.login.config= "
            "-Djavax.security.auth.useSubjectCredsOnly=false")
    .config("spark.driver.extraJavaOptions",
            "-Djava.security.auth.login.config= "
            "-Djavax.security.auth.useSubjectCredsOnly=false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f'Spark: {spark.version}')
print('SparkSession created successfully ✅')

Spark: 3.5.1
SparkSession created successfully ✅


## 6. Kafka / Event Hub Configuration

One kafkaConf dict per Event Hub. Both use the required security settings:

- `kafka.bootstrap.servers`: namespace Kafka endpoint on port 9093
- `kafka.sasl.mechanism`: `PLAIN`
- `kafka.security.protocol`: `SASL_SSL`
- `kafka.sasl.jaas.config`: `PlainLoginModule` with the consumer connection string as the password

`startingOffsets: latest` is used for the initial run. On subsequent runs, Spark automatically resumes from the last committed offset stored in the checkpoint directory, ignoring this setting entirely.
`failOnDataLoss: false` ensures Spark skips missing offsets rather than crashing if the Event Hub retention window has moved past stored checkpoints.

In [16]:
# Spark Structured Streaming manages consumer offsets internally.
# groupIdPrefix auto-generates consumer group IDs used by Spark internally;
# these do NOT appear as named consumer groups in Azure Portal.
# enable.auto.commit is overridden by Spark — listed here to match the
# reference notebook pattern.
# failOnDataLoss=false: if Event Hub ages out old offsets (retention window),
# Spark skips the missing data instead of crashing the query.

kafkaConf_orders = {
    "kafka.bootstrap.servers": f"{event_hub_namespace}.servicebus.windows.net:9093",
    "kafka.sasl.mechanism":    "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config":  (
        f'org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="$ConnectionString" password="{orders_consumer_conn_str}";'
    ),
    "subscribe":               orders_eventhub_name,
    "startingOffsets":         "latest",
    "failOnDataLoss":          "false",
    "enable.auto.commit":      "true",
    "groupIdPrefix":           "Group1_Orders_",
    "auto.commit.interval.ms": "5000"
}

import pprint
pprint.pprint(kafkaConf_orders)


{'auto.commit.interval.ms': '5000',
 'enable.auto.commit': 'true',
 'failOnDataLoss': 'false',
 'groupIdPrefix': 'Group1_Orders_',
 'kafka.bootstrap.servers': 'iesstsabbadbab-grp-01-05.servicebus.windows.net:9093',
 'kafka.sasl.jaas.config': 'org.apache.kafka.common.security.plain.PlainLoginModule '
                           'required username="$ConnectionString" '
                           'password="Endpoint=sb://iesstsabbadbab-grp-01-05.servicebus.windows.net/;SharedAccessKeyName=Consumer;SharedAccessKey=Rex4xptVTzkHMrWp17loxo6RWM8h76qWc+AEhO4uEWU=;EntityPath=group_01_orders";',
 'kafka.sasl.mechanism': 'PLAIN',
 'kafka.security.protocol': 'SASL_SSL',
 'startingOffsets': 'latest',
 'subscribe': 'group_01_orders'}


In [17]:
kafkaConf_courier = {
    "kafka.bootstrap.servers": f"{event_hub_namespace}.servicebus.windows.net:9093",
    "kafka.sasl.mechanism":    "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config":  (
        f'org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="$ConnectionString" password="{courier_consumer_conn_str}";'
    ),
    "subscribe":               courier_eventhub_name,
    "startingOffsets":         "latest",
    "failOnDataLoss":          "false",
    "enable.auto.commit":      "true",
    "groupIdPrefix":           "Group1_Courier_",
    "auto.commit.interval.ms": "5000"
}

pprint.pprint(kafkaConf_courier)


{'auto.commit.interval.ms': '5000',
 'enable.auto.commit': 'true',
 'failOnDataLoss': 'false',
 'groupIdPrefix': 'Group1_Courier_',
 'kafka.bootstrap.servers': 'iesstsabbadbab-grp-01-05.servicebus.windows.net:9093',
 'kafka.sasl.jaas.config': 'org.apache.kafka.common.security.plain.PlainLoginModule '
                           'required username="$ConnectionString" '
                           'password="Endpoint=sb://iesstsabbadbab-grp-01-05.servicebus.windows.net/;SharedAccessKeyName=Consumer;SharedAccessKey=ahLAAZ4MaTdsp0lojerR+wbCcwIl7JN1d+AEhBHL7Zk=;EntityPath=group_01_courier_state_events";',
 'kafka.sasl.mechanism': 'PLAIN',
 'kafka.security.protocol': 'SASL_SSL',
 'startingOffsets': 'latest',
 'subscribe': 'group_01_courier_state_events'}


## 7. Read & Deserialise Streams

Both streams are read with `readStream.format("kafka")`. The raw `value` column contains schemaless AVRO bytes produced by the Python producers. `from_avro()` deserialises them using the schemas defined in Section 2.

`timestamp-millis` fields are cast from `long` to Spark `timestamp` for human-readable display and time-based analytics.

In [18]:
raw_orders = (
    spark.readStream
    .format("kafka")
    .options(**kafkaConf_orders)
    .load()
)

orders_parsed = raw_orders.select(
    from_avro(F.col("value"), order_schema_str, {"mode": "PERMISSIVE"}).alias("order")
)

orders_df = orders_parsed.select(
    F.col("order.schema_version").alias("schema_version"),
    F.col("order.event_id").alias("event_id"),
    F.col("order.event_sequence").alias("event_sequence"),
    F.col("order.event_type").alias("event_type"),
    F.col("order.order_id").alias("order_id"),
    F.col("order.restaurant_id").alias("restaurant_id"),
    F.col("order.zone_id").alias("zone_id"),
    F.col("order.courier_id").alias("courier_id"),
    F.col("order.order_value").alias("order_value"),
    (F.col("order.event_time")  / 1000).cast("timestamp").alias("event_time"),
    (F.col("order.ingest_time") / 1000).cast("timestamp").alias("ingest_time")
)

orders_df.printSchema()

root
 |-- schema_version: integer (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_sequence: integer (nullable = true)
 |-- event_type: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- restaurant_id: string (nullable = true)
 |-- zone_id: string (nullable = true)
 |-- courier_id: string (nullable = true)
 |-- order_value: double (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- ingest_time: timestamp (nullable = true)



In [19]:
raw_courier = (
    spark.readStream
    .format("kafka")
    .options(**kafkaConf_courier)
    .load()
)

courier_parsed = raw_courier.select(
    from_avro(F.col("value"), courier_schema_str, {"mode": "PERMISSIVE"}).alias("courier")
)

courier_df = courier_parsed.select(
    F.col("courier.schema_version").alias("schema_version"),
    F.col("courier.event_id").alias("event_id"),
    F.col("courier.event_type").alias("event_type"),
    F.col("courier.courier_id").alias("courier_id"),
    F.col("courier.order_id").alias("order_id"),
    F.col("courier.zone_id").alias("zone_id"),
    F.col("courier.latitude").alias("latitude"),
    F.col("courier.longitude").alias("longitude"),
    (F.col("courier.event_time")  / 1000).cast("timestamp").alias("event_time"),
    (F.col("courier.ingest_time") / 1000).cast("timestamp").alias("ingest_time")
)

courier_df.printSchema()

root
 |-- schema_version: integer (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- courier_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- zone_id: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- ingest_time: timestamp (nullable = true)



## 8. Analytical Queries (In-Memory Sink)

Queries write to the `memory` sink so results can be inspected interactively with `spark.sql()`.

| Query name | Stream | Business question |
|---|---|---|
| `all_orders` | Orders | Raw event feed for inspection |
| `orders_by_zone` | Orders | Demand distribution across zones |
| `event_type_counts` | Orders | Lifecycle stage breakdown |
| `cancellations` | Orders | Cancelled orders only |
| `avg_order_value` | Orders | Revenue metrics per zone |
| `all_courier_events` | Courier | Raw courier event feed |
| `courier_activity_by_zone` | Courier | Courier density per zone |
| `orders_windowed` | Orders | Order volume + avg value per zone, 5-min sliding window |
| `courier_windowed` | Courier | Active couriers per zone, 5-min sliding window |

In [20]:
q_all_orders = (
        orders_df.writeStream
        .outputMode("append")
        .format("memory")
        .queryName("all_orders")
        .start()
    )

print('Query started: all_orders')

Query started: all_orders


In [21]:
orders_by_zone_df = (
        orders_df
        .filter(F.col("event_type") == "ORDER_CREATED")
        .groupBy("zone_id")
        .agg(F.count("*").alias("order_count"))
    )

q_orders_by_zone = (
        orders_by_zone_df.writeStream
        .outputMode("complete")
        .format("memory")
        .queryName("orders_by_zone")
        .start()
    )

print('Query started: orders_by_zone')

Query started: orders_by_zone


In [22]:
event_type_counts_df = (
        orders_df
        .groupBy("event_type")
        .agg(F.count("*").alias("event_count"))
    )

q_event_type_counts = (
        event_type_counts_df.writeStream
        .outputMode("complete")
        .format("memory")
        .queryName("event_type_counts")
        .start()
    )

print('Query started: event_type_counts')

Query started: event_type_counts


In [23]:
cancellations_df = orders_df.filter(F.col("event_type") == "CANCELLED")

q_cancellations = (
    cancellations_df.writeStream
    .outputMode("append")
    .format("memory")
    .queryName("cancellations")
    .start()
)

print('Query started: cancellations')

Query started: cancellations


In [24]:
avg_order_value_df = (
        orders_df
        .filter(F.col("event_type") == "ORDER_CREATED")
        .groupBy("zone_id")
        .agg(
            F.avg("order_value").alias("avg_order_value"),
            F.min("order_value").alias("min_order_value"),
            F.max("order_value").alias("max_order_value"),
            F.count("*").alias("order_count")
        )
    )

q_avg_order_value = (
        avg_order_value_df.writeStream
        .outputMode("complete")
        .format("memory")
        .queryName("avg_order_value")
        .start()
    )

print('Query started: avg_order_value')

Query started: avg_order_value


In [25]:
q_all_courier = (
        courier_df.writeStream
        .outputMode("append")
        .format("memory")
        .queryName("all_courier_events")
        .start()
    )

print('Query started: all_courier_events')

Query started: all_courier_events


In [26]:
courier_zone_df = (
        courier_df
        .filter(F.col("event_type").isin(["ONLINE", "LOCATION_UPDATE", "ARRIVED_RESTAURANT"]))
        .groupBy("zone_id")
        .agg(
            F.approx_count_distinct("courier_id").alias("active_couriers"),
            F.count("*").alias("event_count")
        )
    )

q_courier_zone = (
        courier_zone_df.writeStream
        .outputMode("complete")
        .format("memory")
        .queryName("courier_activity_by_zone")
        .start()
    )

print('Query started: courier_activity_by_zone')

Query started: courier_activity_by_zone


In [27]:
# Windowed aggregation: orders created per zone per 5-min sliding window

orders_windowed_df = (
    orders_df
    .withWatermark("event_time", "10 minutes")
    .filter(F.col("event_type") == "ORDER_CREATED")
    .groupBy(
        F.window(F.col("event_time"), "5 minutes", "1 minute").alias("window"),
        F.col("zone_id")
    )
    .agg(
        F.count("*").alias("order_count"),
        F.round(F.avg("order_value"), 2).alias("avg_value")
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        F.col("zone_id"),
        F.col("order_count"),
        F.col("avg_value")
    )
)

q_orders_windowed = (
    orders_windowed_df.writeStream
    .outputMode("update")
    .format("memory")
    .queryName("orders_windowed")
    .start()
)

print("Query started: orders_windowed")

Query started: orders_windowed


In [28]:
courier_windowed_df = (
    courier_df
    .withWatermark("event_time", "10 minutes")
    .filter(F.col("event_type").isin(["ONLINE", "LOCATION_UPDATE", "ARRIVED_RESTAURANT"]))
    .groupBy(
        F.window(F.col("event_time"), "5 minutes", "1 minute").alias("window"),
        F.col("zone_id")
    )
    .agg(
        F.approx_count_distinct("courier_id").alias("active_couriers"),
        F.count("*").alias("event_count")
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        F.col("zone_id"),
        F.col("active_couriers"),
        F.col("event_count")
    )
)

q_courier_windowed = (
    courier_windowed_df.writeStream
    .outputMode("update")
    .format("memory")
    .queryName("courier_windowed")
    .start()
)

print("Query started: courier_windowed")

Query started: courier_windowed


In [29]:
for q in spark.streams.active:
    print(f'  [{q.name}]  active={q.isActive}  status={q.status["message"]}')

print()
spark.sql('SHOW TABLES').show(truncate=False)

  [courier_windowed]  active=True  status=Getting offsets from KafkaV2[Subscribe[group_01_courier_state_events]]
  [cancellations]  active=True  status=Getting offsets from KafkaV2[Subscribe[group_01_orders]]
  [orders_windowed]  active=True  status=Getting offsets from KafkaV2[Subscribe[group_01_orders]]
  [avg_order_value]  active=True  status=Getting offsets from KafkaV2[Subscribe[group_01_orders]]
  [all_orders]  active=True  status=Getting offsets from KafkaV2[Subscribe[group_01_orders]]
  [courier_activity_by_zone]  active=True  status=Getting offsets from KafkaV2[Subscribe[group_01_courier_state_events]]
  [all_courier_events]  active=True  status=Getting offsets from KafkaV2[Subscribe[group_01_courier_state_events]]
  [event_type_counts]  active=True  status=Getting offsets from KafkaV2[Subscribe[group_01_orders]]
  [orders_by_zone]  active=True  status=Getting offsets from KafkaV2[Subscribe[group_01_orders]]

+---------+------------------------+-----------+
|namespace|tableNam

In [30]:
!sleep 20
print('Ready to query.')

Ready to query.


## 9. Inspect Query Results

In [31]:
import time

print("Waiting 30s for stream to initialize...")
time.sleep(30)

Waiting 30s for stream to initialize...


### 9.1 Stateless Query Results

In [32]:
print("\nSample Order Events:")
spark.sql(
    'SELECT event_type, zone_id, order_id, ROUND(order_value,2) AS order_value, event_time '
    'FROM all_orders ORDER BY event_time DESC LIMIT 20'
).show(truncate=False)


Sample Order Events:
+----------+-------+------------------------------------+-----------+-----------------------+
|event_type|zone_id|order_id                            |order_value|event_time             |
+----------+-------+------------------------------------+-----------+-----------------------+
|DELIVERED |zone_2 |4a72f8b0-cfa1-45c3-8714-9aaba726f5e2|29.39      |2026-04-19 19:05:37.323|
|DELIVERED |zone_2 |b53f10ae-eba1-4ced-9b83-e703c1b7abb0|11.91      |2026-04-19 18:52:38.552|
|DELIVERED |zone_2 |8c9566b3-4a03-4d42-b3d3-86941edc7276|11.58      |2026-04-19 18:28:09.158|
|DELIVERED |zone_2 |a5072c07-bb8c-4cb0-b26c-e9cd2edee8d8|19.9       |2026-04-19 18:25:09.158|
|DELIVERED |zone_3 |54b20aea-8791-42d8-9f91-0d6da2452bf0|9.87       |2026-04-19 17:37:38.552|
|DELIVERED |zone_1 |f130a028-1179-468a-84e7-dc62834a8b0e|8.0        |2026-04-19 17:36:03.048|
|DELIVERED |zone_1 |b1fa25b0-cbd3-4702-96ce-a16e747b3d54|20.34      |2026-04-19 17:35:58.123|
|DELIVERED |zone_4 |639a360d-5006-45ae

In [33]:
print("\nOrders by Zone:")
spark.sql('SELECT zone_id, order_count FROM orders_by_zone ORDER BY order_count DESC').show()


Orders by Zone:
+-------+-----------+
|zone_id|order_count|
+-------+-----------+
| zone_4|         69|
| zone_2|         67|
| zone_1|         41|
| zone_0|         40|
| zone_3|         33|
+-------+-----------+



In [34]:
print("\nEvent Type Distribution:")
spark.sql('SELECT event_type, event_count FROM event_type_counts ORDER BY event_count DESC').show()


Event Type Distribution:
+-------------------+-----------+
|         event_type|event_count|
+-------------------+-----------+
|RESTAURANT_ACCEPTED|        328|
|      ORDER_CREATED|        327|
|       PREP_STARTED|        312|
|   COURIER_ASSIGNED|        291|
|          DELIVERED|        289|
|          PREP_DONE|        288|
|          PICKED_UP|        266|
|          CANCELLED|         40|
+-------------------+-----------+



In [35]:
print("\nCancellations:")
spark.sql(
    'SELECT order_id, zone_id, restaurant_id, event_time '
    'FROM cancellations ORDER BY event_time DESC LIMIT 20'
).show(truncate=False)


Cancellations:
+------------------------------------+-------+--------------+-----------------------+
|order_id                            |zone_id|restaurant_id |event_time             |
+------------------------------------+-------+--------------+-----------------------+
|499b6bd6-b067-4a2d-9d5d-890b1377cedc|zone_0 |rest_zone_0_11|2026-04-19 16:46:09.158|
|36d8dc60-b436-41c1-aa77-7001b2da2f77|zone_2 |rest_zone_2_10|2026-04-19 16:45:56.902|
|36d8dc60-b436-41c1-aa77-7001b2da2f77|zone_2 |rest_zone_2_10|2026-04-19 16:45:56.902|
|45b287a2-9943-4b7a-bf89-1988501972a4|zone_3 |rest_zone_3_11|2026-04-19 16:45:50.788|
|8967d7ad-1f2d-4ee0-a3c6-891b4ebd05fa|zone_0 |rest_zone_0_4 |2026-04-19 16:45:47.107|
|931ab7d0-0a2e-44d1-b34d-2c16ffd05316|zone_0 |rest_zone_0_4 |2026-04-19 16:45:42.217|
|af736839-7d3f-4ee6-801f-4c0f909c2145|zone_2 |rest_zone_2_7 |2026-04-19 16:45:41.09 |
|cccbccb6-8690-4abe-952a-8786e0471a22|zone_4 |rest_zone_4_1 |2026-04-19 16:45:36.096|
|e8a52633-b672-4e20-ba0d-5f23614cf763|

In [36]:
print("\nAverage Order Value by Zone:")
spark.sql(
    'SELECT zone_id, ROUND(avg_order_value,2) AS avg_value, '
    'ROUND(min_order_value,2) AS min_value, ROUND(max_order_value,2) AS max_value, '
    'order_count FROM avg_order_value ORDER BY avg_value DESC'
).show()


Average Order Value by Zone:
+-------+---------+---------+---------+-----------+
|zone_id|avg_value|min_value|max_value|order_count|
+-------+---------+---------+---------+-----------+
| zone_3|    26.37|     9.87|    58.75|         51|
| zone_2|    22.17|     8.81|    69.62|        111|
| zone_0|     22.0|      8.0|    56.66|         60|
| zone_4|    21.52|     8.29|    63.35|        108|
| zone_1|    20.94|      8.0|     68.6|         65|
+-------+---------+---------+---------+-----------+



In [37]:
print("\nSample Courier Events:")
spark.sql(
    'SELECT event_type, courier_id, zone_id, latitude, longitude, event_time '
    'FROM all_courier_events ORDER BY event_time DESC LIMIT 20'
).show(truncate=False)


Sample Courier Events:
+----------+----------+-------+--------+---------+-----------------------+
|event_type|courier_id|zone_id|latitude|longitude|event_time             |
+----------+----------+-------+--------+---------+-----------------------+
|ONLINE    |courier_31|zone_4 |NULL    |NULL     |2026-04-19 17:10:26.347|
|ONLINE    |courier_10|zone_4 |NULL    |NULL     |2026-04-19 17:10:10.845|
|ONLINE    |courier_7 |zone_1 |NULL    |NULL     |2026-04-19 17:08:49.491|
|ONLINE    |courier_7 |zone_1 |NULL    |NULL     |2026-04-19 17:08:49.491|
|ONLINE    |courier_36|zone_4 |NULL    |NULL     |2026-04-19 17:07:38.282|
|ONLINE    |courier_34|zone_2 |NULL    |NULL     |2026-04-19 17:07:26.863|
|ONLINE    |courier_23|zone_2 |NULL    |NULL     |2026-04-19 17:06:40.064|
|ONLINE    |courier_23|zone_2 |NULL    |NULL     |2026-04-19 17:06:40.064|
|ONLINE    |courier_1 |zone_1 |NULL    |NULL     |2026-04-19 17:05:15.157|
|ONLINE    |courier_21|zone_4 |NULL    |NULL     |2026-04-19 17:05:00.158|
|

In [38]:
print("\nCourier Activity by Zone:")
spark.sql(
    'SELECT zone_id, active_couriers, event_count '
    'FROM courier_activity_by_zone ORDER BY active_couriers DESC'
).show()


Courier Activity by Zone:
+-------+---------------+-----------+
|zone_id|active_couriers|event_count|
+-------+---------------+-----------+
| zone_4|             14|        317|
| zone_1|             11|        308|
| zone_2|             10|        269|
| zone_0|              9|        246|
| zone_3|              6|        190|
+-------+---------------+-----------+



### 9.2 Windowed Query Results

The following queries inspect the windowed aggregations — results are bucketed
by 5-minute sliding windows based on event_time, with a 10-minute watermark
handling late-arriving events.

In [39]:
print("\nWindowed Order Volume (last 10 windows):")
spark.sql("""
    SELECT
        window_start,
        window_end,
        zone_id,
        order_count,
        avg_value
    FROM orders_windowed
    ORDER BY window_start DESC, order_count DESC
    LIMIT 30
""").show(truncate=False)


Windowed Order Volume (last 10 windows):
+-------------------+-------------------+-------+-----------+---------+
|window_start       |window_end         |zone_id|order_count|avg_value|
+-------------------+-------------------+-------+-----------+---------+
|2026-04-19 16:43:00|2026-04-19 16:48:00|zone_2 |24         |17.67    |
|2026-04-19 16:43:00|2026-04-19 16:48:00|zone_4 |17         |24.37    |
|2026-04-19 16:43:00|2026-04-19 16:48:00|zone_0 |14         |20.44    |
|2026-04-19 16:43:00|2026-04-19 16:48:00|zone_1 |13         |19.49    |
|2026-04-19 16:43:00|2026-04-19 16:48:00|zone_2 |9          |16.11    |
|2026-04-19 16:43:00|2026-04-19 16:48:00|zone_3 |9          |20.4     |
|2026-04-19 16:43:00|2026-04-19 16:48:00|zone_4 |7          |20.67    |
|2026-04-19 16:43:00|2026-04-19 16:48:00|zone_0 |5          |27.05    |
|2026-04-19 16:43:00|2026-04-19 16:48:00|zone_3 |2          |33.1     |
|2026-04-19 16:43:00|2026-04-19 16:48:00|zone_1 |2          |14.85    |
|2026-04-19 16:42:00|2

In [40]:
print("\nWindowed Courier Activity (last 10 windows):")
spark.sql("""
    SELECT
        window_start,
        window_end,
        zone_id,
        active_couriers,
        event_count
    FROM courier_windowed
    ORDER BY window_start DESC, active_couriers DESC
    LIMIT 30
""").show(truncate=False)


Windowed Courier Activity (last 10 windows):
+-------------------+-------------------+-------+---------------+-----------+
|window_start       |window_end         |zone_id|active_couriers|event_count|
+-------------------+-------------------+-------+---------------+-----------+
|2026-04-19 17:13:00|2026-04-19 17:18:00|zone_0 |1              |1          |
|2026-04-19 17:12:00|2026-04-19 17:17:00|zone_0 |1              |1          |
|2026-04-19 17:11:00|2026-04-19 17:16:00|zone_0 |1              |1          |
|2026-04-19 17:10:00|2026-04-19 17:15:00|zone_4 |2              |2          |
|2026-04-19 17:10:00|2026-04-19 17:15:00|zone_4 |1              |1          |
|2026-04-19 17:10:00|2026-04-19 17:15:00|zone_3 |1              |1          |
|2026-04-19 17:10:00|2026-04-19 17:15:00|zone_0 |1              |1          |
|2026-04-19 17:09:00|2026-04-19 17:14:00|zone_4 |2              |2          |
|2026-04-19 17:09:00|2026-04-19 17:14:00|zone_4 |1              |1          |
|2026-04-19 17:09:

### 9.3 Stop sinks

In [41]:
# Stop memory sinks now that inspection is done — free up resources for blob writes
memory_sinks = ["all_orders", "all_courier_events", "orders_by_zone",
                "event_type_counts", "cancellations", "avg_order_value",
                "courier_activity_by_zone", "orders_windowed", "courier_windowed"]

for q in spark.streams.active:
    if q.name in memory_sinks:
        q.stop()
        print(f"Stopped: {q.name}")
print("Memory sinks stopped. Blob streams continue running.")

Stopped: courier_windowed
Stopped: cancellations
Stopped: orders_windowed
Stopped: avg_order_value
Stopped: all_orders
Stopped: courier_activity_by_zone
Stopped: all_courier_events
Stopped: event_type_counts
Stopped: orders_by_zone
Memory sinks stopped. Blob streams continue running.


## 10. Write Streams to Azure Blob Storage (Parquet)

Both streams are persisted to Azure Blob Storage in Parquet format via the `wasbs://` connector.

- Checkpoint directories are stored in Blob for fault-tolerant exactly-once semantics.
- Trigger is set to 10 seconds so micro-batches land frequently.
- Data in Blob can be consumed by downstream tools (Power BI, Streamlit, ML pipelines).
- Windowed aggregations (orders_windowed/, courier_windowed/)
  are written using foreachBatch with append mode. Only
  finalized windows land in storage.

In [42]:
base_path = f"wasbs://{container_name}@{account_name}.blob.core.windows.net"

orders_output_path      = f"{base_path}/orders/"
orders_checkpoint_path  = f"{base_path}/checkpoints/orders/"

courier_output_path     = f"{base_path}/courier_state_events/"
courier_checkpoint_path = f"{base_path}/checkpoints/courier_state_events/"

print(f'Orders output     : {orders_output_path}')
print(f'Orders checkpoint : {orders_checkpoint_path}')
print(f'Courier output    : {courier_output_path}')
print(f'Courier checkpoint: {courier_checkpoint_path}')

Orders output     : wasbs://group01output@iesstsabbadbab.blob.core.windows.net/orders/
Orders checkpoint : wasbs://group01output@iesstsabbadbab.blob.core.windows.net/checkpoints/orders/
Courier output    : wasbs://group01output@iesstsabbadbab.blob.core.windows.net/courier_state_events/
Courier checkpoint: wasbs://group01output@iesstsabbadbab.blob.core.windows.net/checkpoints/courier_state_events/


In [43]:
q_orders_blob = (
    orders_df.writeStream
    .format("parquet")
    .option("checkpointLocation", orders_checkpoint_path)
    .option("path", orders_output_path)
    .queryName("orders_to_blob")
    .trigger(processingTime="10 seconds")
    .start()
)
print(f'Streaming orders -> {orders_output_path}')

Streaming orders -> wasbs://group01output@iesstsabbadbab.blob.core.windows.net/orders/


In [44]:
q_courier_blob = (
    courier_df.writeStream
    .format("parquet")
    .option("checkpointLocation", courier_checkpoint_path)
    .option("path", courier_output_path)
    .queryName("courier_to_blob")
    .trigger(processingTime="10 seconds")
    .start()
)
print(f'Streaming courier events -> {courier_output_path}')

Streaming courier events -> wasbs://group01output@iesstsabbadbab.blob.core.windows.net/courier_state_events/


In [45]:
# Write analytical aggregations to Blob Storage as Parquet (data at rest)
from pyspark.sql import functions as F

def write_batch(df, epoch_id, path):
    if df.count() > 0:
        df.write.mode("overwrite").parquet(path)

q_zone_demand_blob = (
    orders_by_zone_df.writeStream
    .outputMode("complete")
    .foreachBatch(lambda df, epoch_id: write_batch(df, epoch_id, f"{base_path}/analytics/zone_demand/"))
    .queryName("zone_demand_to_blob")
    .trigger(processingTime="10 seconds")
    .start()
)

q_cancellations_blob = (
    cancellations_df.writeStream
    .outputMode("append")
    .format("parquet")
    .option("checkpointLocation", f"{base_path}/checkpoints/cancellations/")
    .option("path", f"{base_path}/analytics/cancellations/")
    .queryName("cancellations_to_blob")
    .trigger(processingTime="10 seconds")
    .start()
)

q_avg_value_blob = (
    avg_order_value_df.writeStream
    .outputMode("complete")
    .foreachBatch(lambda df, epoch_id: write_batch(df, epoch_id, f"{base_path}/analytics/avg_order_value/"))
    .queryName("avg_order_value_to_blob")
    .trigger(processingTime="10 seconds")
    .start()
)

print("Analytical aggregations streaming to Blob Storage.")

Analytical aggregations streaming to Blob Storage.


In [46]:
q_orders_windowed_blob = (
    orders_windowed_df.writeStream
    .outputMode("append")
    .foreachBatch(lambda df, epoch_id: write_batch(df, epoch_id, f"{base_path}/analytics/orders_windowed/"))
    .queryName("orders_windowed_to_blob")
    .trigger(processingTime="10 seconds")
    .start()
)

q_courier_windowed_blob = (
    courier_windowed_df.writeStream
    .outputMode("append")
    .foreachBatch(lambda df, epoch_id: write_batch(df, epoch_id, f"{base_path}/analytics/courier_windowed/"))
    .queryName("courier_windowed_to_blob")
    .trigger(processingTime="10 seconds")
    .start()
)

print("Windowed aggregations streaming to Blob Storage.")

Windowed aggregations streaming to Blob Storage.


In [47]:
for q in spark.streams.active:
    print(f'[{q.name}]  active={q.isActive}  status={q.status["message"]}')

[courier_windowed_to_blob]  active=True  status=Getting offsets from KafkaV2[Subscribe[group_01_courier_state_events]]
[cancellations_to_blob]  active=True  status=Initializing sources
[orders_to_blob]  active=True  status=Processing new data
[orders_windowed_to_blob]  active=True  status=Getting offsets from KafkaV2[Subscribe[group_01_orders]]
[courier_to_blob]  active=True  status=Processing new data
[avg_order_value_to_blob]  active=True  status=Getting offsets from KafkaV2[Subscribe[group_01_orders]]
[zone_demand_to_blob]  active=True  status=Processing new data


In [48]:
for q in [q_orders_blob, q_courier_blob]:
    prog = q.lastProgress
    if prog:
        print(f'\n=== {q.name} ===')
        print(f'  Batch ID   : {prog["batchId"]}')
        print(f'  Input rows : {prog["numInputRows"]}')
        print(f'  Timestamp  : {prog["timestamp"]}')
        print(f'  Sink       : {prog["sink"]}')


In [49]:
!sleep 30
print('Parquet files should now be in Azure Blob Storage.')

Parquet files should now be in Azure Blob Storage.


## 11. Read Back from Azure Blob Storage

Verify the data landed correctly by reading the Parquet files back as static DataFrames.

Rather than reading the entire orders/ and courier_state_events/ folders via Spark (which does a full scan of all accumulated files and is extremely slow),
we use the Azure Blob SDK to take the 5 most recently written Parquet files per topic, and load them as pandas DataFrames. This gives an instant snapshot of the latest data at rest.

In [50]:
from azure.storage.blob import BlobServiceClient
import io

client = BlobServiceClient(
    account_url=f"https://{account_name}.blob.core.windows.net",
    credential=account_key
)
container = client.get_container_client(container_name)

# Get 5 most recent orders parquet files
blobs = sorted(
    [b for b in container.list_blobs(name_starts_with="orders/")
     if b.name.endswith(".parquet")],
    key=lambda x: x.last_modified,
    reverse=True
)[:5]

print(f"Total Parquet files in orders/: {len(list(container.list_blobs(name_starts_with='orders/')))}")
print(f"Reading {len(blobs)} most recent files for verification...\n")

import pandas as pd
dfs = []
for b in blobs:
    raw = container.download_blob(b.name).readall()
    dfs.append(pd.read_parquet(io.BytesIO(raw)))

df_sample = pd.concat(dfs, ignore_index=True)
print(f"Events in sample       : {len(df_sample)}")
print(f"\nBy event type:")
print(df_sample['event_type'].value_counts().to_string())
print(f"\nBy zone:")
print(df_sample['zone_id'].value_counts().sort_index().to_string())

Total Parquet files in orders/: 2605
Reading 5 most recent files for verification...

Events in sample       : 11653

By event type:
event_type
ORDER_CREATED          1771
RESTAURANT_ACCEPTED    1765
PREP_STARTED           1655
DELIVERED              1600
COURIER_ASSIGNED       1597
PREP_DONE              1595
PICKED_UP              1514
CANCELLED               156

By zone:
zone_id
zone_0    2033
zone_1    1888
zone_2    3737
zone_3    1424
zone_4    2571


In [51]:
blobs_courier = sorted(
    [b for b in container.list_blobs(name_starts_with="courier_state_events/")
     if b.name.endswith(".parquet")],
    key=lambda x: x.last_modified,
    reverse=True
)[:5]

print(f"Total Parquet files in courier_state_events/: {len(list(container.list_blobs(name_starts_with='courier_state_events/')))}")
print(f"Reading {len(blobs_courier)} most recent files for verification...\n")

dfs_courier = []
for b in blobs_courier:
    raw = container.download_blob(b.name).readall()
    dfs_courier.append(pd.read_parquet(io.BytesIO(raw)))

df_courier_sample = pd.concat(dfs_courier, ignore_index=True)
print(f"Events in sample       : {len(df_courier_sample)}")
print(f"\nBy event type:")
print(df_courier_sample['event_type'].value_counts().to_string())
print(f"\nLocation updates with GPS (sample):")
gps = df_courier_sample[df_courier_sample['latitude'].notna()][
    ['courier_id', 'zone_id', 'latitude', 'longitude', 'event_time']
].head(10)
print(gps.to_string(index=False))

Total Parquet files in courier_state_events/: 2604
Reading 5 most recent files for verification...

Events in sample       : 1924

By event type:
event_type
LOCATION_UPDATE       629
ARRIVED_RESTAURANT    590
ARRIVED_CUSTOMER      296
PICKED_UP_ORDER       292
ONLINE                 83
OFFLINE                34

Location updates with GPS (sample):
courier_id zone_id  latitude  longitude              event_time
 courier_4  zone_2 40.038890  -3.661346 2026-04-19 16:03:45.904
courier_23  zone_2 40.040429  -3.655114 2026-04-19 16:08:58.904
courier_23  zone_2 40.040429  -3.655114 2026-04-19 16:08:58.904
 courier_5  zone_4 40.081969  -3.617429 2026-04-19 16:03:45.904
courier_37  zone_1 40.022602  -3.684088 2026-04-19 16:03:45.904
courier_24  zone_4 40.077366  -3.624861 2026-04-19 16:09:38.339
courier_27  zone_2 40.036050  -3.663413 2026-04-19 16:03:47.339
courier_18  zone_4 40.080889  -3.615173 2026-04-19 16:09:45.339
courier_11  zone_2 40.037942  -3.664386 2026-04-19 16:03:50.098
courier_37

In [52]:
# Delivery KPIs from the orders sample
n_created   = (df_sample['event_type'] == 'ORDER_CREATED').sum()
n_cancelled = (df_sample['event_type'] == 'CANCELLED').sum()
n_delivered = (df_sample['event_type'] == 'DELIVERED').sum()

print(f"Orders created   : {n_created}")
print(f"Orders cancelled : {n_cancelled}  ({100*n_cancelled/max(n_created,1):.1f}%)")
print(f"Orders delivered : {n_delivered}  ({100*n_delivered/max(n_created,1):.1f}%)")

Orders created   : 1771
Orders cancelled : 156  (8.8%)
Orders delivered : 1600  (90.3%)


In [53]:
# Verify checkpoints exist in Blob
# exactly-once semantics

from azure.storage.blob import BlobServiceClient

client = BlobServiceClient(
    account_url=f"https://{account_name}.blob.core.windows.net",
    credential=account_key
)
container = client.get_container_client(container_name)

checkpoint_blobs = [
    b.name for b in container.list_blobs(name_starts_with="checkpoints/")
]

print(f"Checkpoint files found: {len(checkpoint_blobs)}")
print("Sample entries:")
for b in checkpoint_blobs[:10]:
    print(f"  {b}")

Checkpoint files found: 647
Sample entries:
  checkpoints/cancellations
  checkpoints/cancellations/commits
  checkpoints/cancellations/commits/713
  checkpoints/cancellations/commits/714
  checkpoints/cancellations/commits/715
  checkpoints/cancellations/commits/716
  checkpoints/cancellations/commits/717
  checkpoints/cancellations/commits/718
  checkpoints/cancellations/commits/719
  checkpoints/cancellations/commits/720


## 12. Pipeline Health

### 12.1 Health Check

In [54]:
print("orders producer alive:", orders_proc.poll() is None)
print("courier producer alive:", courier_proc.poll() is None)

for q in [q_orders_blob, q_courier_blob]:
    print("\n", q.name)
    print("isActive:", q.isActive)
    print("exception:", q.exception())
    print("lastProgress:", q.lastProgress)

orders producer alive: True
courier producer alive: True

 orders_to_blob
isActive: True
exception: None
lastProgress: None

 courier_to_blob
isActive: True
exception: None
lastProgress: None


### 12.2 Graceful shutdown

In [55]:
if False:  # change to True to stop all streaming queries
    for q in spark.streams.active:
        q.stop()
        print(f'Stopped: {q.name}')
    spark.stop()
    print('Spark session stopped.')

In [56]:
if False:  # change to True to stop both producers
    orders_proc.terminate()
    courier_proc.terminate()
    print('Producers stopped.')

## 13. Dashboard view — Streamlit

### 13.1 Monitor running

In [57]:
import time, datetime, threading

def monitor():
    while True:
        now = datetime.datetime.now().strftime("%H:%M:%S")
        active = spark.streams.active
        names = [f"{q.name}({'✅' if q.isActive else '❌'})" for q in active]
        rows  = [q.lastProgress['numInputRows'] if q.lastProgress else 0 for q in active]
        print(f"[{now}] Active streams: {names} | Input rows: {rows}")
        time.sleep(15)

t = threading.Thread(target=monitor, daemon=True)
t.start()
print("Monitor running in background")

Monitor running in background


### 13.2 Dependencies

In [58]:
!pip install -q streamlit pyngrok azure-storage-blob pandas pyarrow plotly streamlit-autorefresh

[16:44:44] Active streams: ['courier_windowed_to_blob(✅)', 'cancellations_to_blob(✅)', 'orders_to_blob(✅)', 'orders_windowed_to_blob(✅)', 'courier_to_blob(✅)', 'avg_order_value_to_blob(✅)', 'zone_demand_to_blob(✅)'] | Input rows: [7, 0, 0, 0, 0, 0, 0]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 700.8/700.8 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 51.0 MB/s eta 0:00:00


### 13.3 app.py

In [59]:
!pkill -f streamlit #kills any Streamlit process that's already running before starting a new one

In [60]:
%%writefile app.py
import io
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st
from azure.storage.blob import BlobServiceClient
from datetime import datetime, timezone

# Configuration
ACCOUNT_NAME   = "iesstsabbadbab"
ACCOUNT_KEY    = "QOlT3Y2sp4wDkdW+Yb3Cddfo9PdoG8+ORBZy0ZDMZCiQtkK+rtI+Ri1WVXopfWgyoqC8N69czF9L+AStbdQ8bA=="
CONTAINER_NAME = "group01output"
ORDERS_PREFIX  = "orders/"
COURIER_PREFIX = "courier_state_events/"
ACCOUNT_URL    = f"https://{ACCOUNT_NAME}.blob.core.windows.net"

# Only reads the N most-recently-written parquet files per topic.
MAX_FILES = 6

ZONE_ORDER = [f"zone_{i}" for i in range(5)]
ORDER_EVENTS  = ["ORDER_CREATED","RESTAURANT_ACCEPTED","PREP_STARTED",
                 "PREP_DONE","COURIER_ASSIGNED","PICKED_UP","DELIVERED","CANCELLED"]
COURIER_EVENTS = ["ONLINE","LOCATION_UPDATE","ARRIVED_RESTAURANT",
                  "PICKED_UP_ORDER","ARRIVED_CUSTOMER","OFFLINE"]
ORDER_COLORS = {
    "ORDER_CREATED":"#38BDF8","RESTAURANT_ACCEPTED":"#34D399",
    "PREP_STARTED":"#A78BFA","PREP_DONE":"#FBBF24",
    "COURIER_ASSIGNED":"#F472B6","PICKED_UP":"#FB923C",
    "DELIVERED":"#4ADE80","CANCELLED":"#F87171",
}

# Page
st.set_page_config(page_title="Group 1 — Live Delivery Dashboard", layout="wide")

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Space+Mono:wght@700&family=DM+Sans:wght@400;600&display=swap');
html, body, [class*="css"] {
    background:#0A0F1E !important; color:#E2E8F0 !important;
    font-family:'DM Sans',sans-serif;
}
.main .block-container { padding:1.5rem 2rem; max-width:1400px; }
.kpi { background:#111827; border:1px solid #1E293B; border-radius:10px;
       padding:1rem 1.3rem; }
.kpi-label { font-size:0.7rem; color:#64748B; text-transform:uppercase;
             letter-spacing:1.2px; font-family:'Space Mono',monospace; }
.kpi-value { font-family:'Space Mono',monospace; font-size:1.9rem;
             font-weight:700; color:#F8FAFC; line-height:1.1; }
.kpi-sub   { font-size:0.72rem; color:#475569; margin-top:3px; }
.dot { display:inline-block; width:8px; height:8px; border-radius:50%;
       background:#4ADE80; box-shadow:0 0 7px #4ADE80;
       animation:pulse 1.8s infinite; margin-right:6px; vertical-align:middle; }
@keyframes pulse { 0%,100%{opacity:1} 50%{opacity:0.3} }
.ts { font-size:0.75rem; color:#475569; font-family:'Space Mono',monospace;
      margin-bottom:1.2rem; }
#MainMenu,footer,header{visibility:hidden}
</style>
""", unsafe_allow_html=True)

# Data
def load_recent(prefix: str, max_files: int = MAX_FILES):
    """
    Read only the MAX_FILES most-recently-modified parquet blobs under prefix.
    This gives a rolling live window instead of an ever-growing cumulative dump.
    BlobServiceClient is NOT cached so every call gets a fresh blob listing.
    """
    try:
        container = BlobServiceClient(
            account_url=ACCOUNT_URL, credential=ACCOUNT_KEY
        ).get_container_client(CONTAINER_NAME)

        blobs = sorted(
            [b for b in container.list_blobs(name_starts_with=prefix)
             if b.name.endswith(".parquet")],
            key=lambda x: x.last_modified,
            reverse=True,
        )
    except Exception as e:
        st.warning(f"Azure error: {e}")
        return pd.DataFrame(), None

    if not blobs:
        return pd.DataFrame(), None

    newest_ts = blobs[0].last_modified
    recent    = blobs[:max_files]

    dfs = []
    for b in recent:
        try:
            raw = container.download_blob(b.name).readall()
            dfs.append(pd.read_parquet(io.BytesIO(raw)))
        except Exception:
            pass

    if not dfs:
        return pd.DataFrame(), newest_ts

    df = pd.concat(dfs, ignore_index=True)

    if "event_time" in df.columns:
        col = df["event_time"]
        if pd.api.types.is_numeric_dtype(col):
            df["event_time"] = pd.to_datetime(col, unit="ms", errors="coerce")
        else:
            df["event_time"] = pd.to_datetime(col, errors="coerce")

    return df, newest_ts

# Helpers
def fmt(ts):
    if ts is None: return "—"
    try: return pd.Timestamp(ts).strftime("%H:%M:%S")
    except: return "—"

def active_couriers(df):
    if df.empty or "courier_id" not in df.columns: return 0
    if "event_type" not in df.columns: return df["courier_id"].nunique()
    latest = (df.dropna(subset=["event_time"])
               .sort_values("event_time")
               .groupby("courier_id", as_index=False).tail(1))
    return int(latest[latest["event_type"] != "OFFLINE"]["courier_id"].nunique())

DARK = dict(
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font=dict(family="DM Sans", color="#94A3B8", size=12),
    margin=dict(l=10,r=10,t=36,b=10),
    xaxis=dict(gridcolor="#1E293B", zerolinecolor="#1E293B"),
    yaxis=dict(gridcolor="#1E293B", zerolinecolor="#1E293B"),
    title_font=dict(family="Space Mono", size=12, color="#CBD5E1"),
)

# Live Fragment
from streamlit_autorefresh import st_autorefresh
st_autorefresh(interval=15_000, key="live_refresh")

def dashboard():
    orders,  o_ts  = load_recent(ORDERS_PREFIX)
    couriers, c_ts = load_recent(COURIER_PREFIX)

    now   = datetime.now(timezone.utc).strftime("%H:%M:%S UTC")
    o_blob = o_ts.strftime("%H:%M:%S") if o_ts else "—"
    c_blob = c_ts.strftime("%H:%M:%S") if c_ts else "—"

    st.markdown(
        f'<h1>🛵 Food Delivery &nbsp;<span style="font-size:0.9rem;color:#64748B">'
        f'<span class="dot"></span>LIVE</span></h1>',
        unsafe_allow_html=True
    )
    st.markdown(
        f'<div class="ts">Refreshes every 15 s &nbsp;·&nbsp; {now} '
        f'&nbsp;·&nbsp; latest orders blob: {o_blob} '
        f'&nbsp;·&nbsp; latest courier blob: {c_blob}</div>',
        unsafe_allow_html=True
    )

    if orders.empty and couriers.empty:
        st.warning("No data yet — Spark hasn't written any Parquet files. "
                   "Make sure the blob-write queries are running.")
        return

    # KPIs
    n_created   = int((orders["event_type"] == "ORDER_CREATED").sum())  if not orders.empty and "event_type" in orders.columns else 0
    n_delivered = int((orders["event_type"] == "DELIVERED").sum())      if not orders.empty and "event_type" in orders.columns else 0
    n_cancelled = int((orders["event_type"] == "CANCELLED").sum())      if not orders.empty and "event_type" in orders.columns else 0
    cancel_pct  = f"{100*n_cancelled/n_created:.1f}%" if n_created > 0 else "—"
    active_now  = active_couriers(couriers)
    latest_o    = fmt(orders["event_time"].max()   if not orders.empty  and "event_time" in orders.columns else None)
    latest_c    = fmt(couriers["event_time"].max() if not couriers.empty and "event_time" in couriers.columns else None)

    k1, k2, k3, k4, k5 = st.columns(5)
    for col, label, value, sub, accent in [
        (k1, "Orders created",  f"{n_created:,}",    f"latest @ {latest_o}",                        "#38BDF8"),
        (k2, "Delivered",       f"{n_delivered:,}",  f"{100*n_delivered/max(n_created,1):.0f}% of created", "#4ADE80"),
        (k3, "Cancelled",       f"{n_cancelled:,}",  f"cancel rate {cancel_pct}",                   "#F87171"),
        (k4, "Active couriers", f"{active_now}",     f"latest @ {latest_c}",                        "#FBBF24"),
        (k5, "Rolling window",  f"{MAX_FILES} files",f"~{MAX_FILES*10}s of live data",              "#A78BFA"),
    ]:
        col.markdown(
            f'<div class="kpi" style="border-top:3px solid {accent}">'
            f'<div class="kpi-label">{label}</div>'
            f'<div class="kpi-value">{value}</div>'
            f'<div class="kpi-sub">{sub}</div></div>',
            unsafe_allow_html=True
        )

    st.markdown("<br>", unsafe_allow_html=True)

    # Row 1: orders by zone, event mix donut
    c1, c2 = st.columns([1.5, 1])
    with c1:
        if not orders.empty and "zone_id" in orders.columns:
            zc = (orders.groupby("zone_id").size()
                  .reindex(ZONE_ORDER, fill_value=0).reset_index(name="n"))
            fig = px.bar(zc, x="zone_id", y="n", title="Orders by zone",
                         category_orders={"zone_id": ZONE_ORDER})
            fig.update_traces(marker_color="#38BDF8", marker_line_width=0)
            fig.update_layout(**DARK)
            st.plotly_chart(fig, use_container_width=True, config={"displayModeBar":False})

    with c2:
        if not orders.empty and "event_type" in orders.columns:
            em = orders.groupby("event_type").size().reset_index(name="n")
            fig = px.pie(em, names="event_type", values="n", hole=0.45,
                         title="Event mix",
                         color="event_type", color_discrete_map=ORDER_COLORS)
            fig.update_traces(textposition="outside", textinfo="label+percent",
                              textfont_size=10)
            fig.update_layout(**DARK)
            st.plotly_chart(fig, use_container_width=True, config={"displayModeBar":False})

    # Row 2: delivery funnel, avg order value
    c3, c4 = st.columns(2)
    with c3:
        if not orders.empty and {"event_type", "order_id"}.issubset(orders.columns):
            stages = [
                "ORDER_CREATED",
                "RESTAURANT_ACCEPTED",
                "PREP_STARTED",
                "PREP_DONE",
                "COURIER_ASSIGNED",
                "PICKED_UP",
                "DELIVERED",
            ]

            stage_df = (
                orders[orders["event_type"].isin(stages)][["order_id", "event_type"]]
                .dropna()
                .drop_duplicates(["order_id", "event_type"])
            )

            counts = (
                stage_df.groupby("event_type")["order_id"]
                .nunique()
                .reindex(stages, fill_value=0)
            )

            pairs = [(stage, int(counts.loc[stage])) for stage in stages if counts.loc[stage] > 0]

            if pairs:
                fig = go.Figure(go.Funnel(
                    y=[x[0] for x in pairs],
                    x=[x[1] for x in pairs],
                    textinfo="value+percent initial",
                    marker=dict(
                        color=["#38BDF8", "#34D399", "#A78BFA",
                              "#FBBF24", "#F472B6", "#FB923C", "#4ADE80"][:len(pairs)],
                        line=dict(width=0)
                    ),
                ))
                fig.update_layout(
                    title="Order lifecycle funnel (unique orders)",
                    **{**DARK, "margin": dict(l=160, r=10, t=36, b=10)}
                )
                st.plotly_chart(fig, use_container_width=True, config={"displayModeBar": False})

    with c4:
        if not orders.empty and "order_value" in orders.columns and "event_type" in orders.columns:
            sub = orders[orders["event_type"]=="ORDER_CREATED"].dropna(subset=["order_value"])
            if not sub.empty:
                agg = (sub.groupby("zone_id")["order_value"]
                       .agg(avg="mean", min="min", max="max")
                       .reset_index().sort_values("zone_id"))
                fig = go.Figure()
                for name, col_key, color in [("Avg","avg","#FBBF24"),
                                              ("Min","min","#38BDF8"),
                                              ("Max","max","#F472B6")]:
                    fig.add_trace(go.Bar(name=f"{name} €", x=agg["zone_id"],
                                         y=agg[col_key].round(2),
                                         marker_color=color, marker_line_width=0))
                fig.update_layout(barmode="group", title="Order value by zone (€)", **DARK)
                st.plotly_chart(fig, use_container_width=True, config={"displayModeBar":False})

    # Row 3: courier events, courier heatmap
    c5, c6 = st.columns([1, 1.5])
    with c5:
        if not couriers.empty and "event_type" in couriers.columns:
            ce = (couriers.groupby("event_type").size()
                  .reindex(COURIER_EVENTS, fill_value=0).reset_index(name="n"))
            fig = px.bar(ce, x="event_type", y="n", title="Courier event counts",
                         category_orders={"event_type": COURIER_EVENTS})
            fig.update_traces(marker_color="#34D399", marker_line_width=0)
            fig.update_layout(**DARK)
            fig.update_xaxes(tickangle=-30)
            st.plotly_chart(fig, use_container_width=True, config={"displayModeBar":False})

    with c6:
        if not couriers.empty and "zone_id" in couriers.columns and "event_type" in couriers.columns:
            pivot = (couriers.groupby(["zone_id","event_type"]).size()
                     .reset_index(name="n")
                     .pivot(index="zone_id", columns="event_type", values="n")
                     .fillna(0))
            fig = go.Figure(go.Heatmap(
                z=pivot.values, x=pivot.columns.tolist(),
                y=pivot.index.tolist(), colorscale="Teal",
            ))
            fig.update_layout(title="Courier events: zone × type",
                              **{**DARK, "margin":dict(l=70,r=10,t=36,b=60)})
            fig.update_xaxes(tickangle=-30)
            st.plotly_chart(fig, use_container_width=True, config={"displayModeBar":False})

    # Raw tables
    with st.expander("Raw orders sample (newest 50)"):
        if not orders.empty:
            st.dataframe(orders.sort_values("event_time", ascending=False).head(50),
                         use_container_width=True)
    with st.expander("Raw courier sample (newest 50)"):
        if not couriers.empty:
            st.dataframe(couriers.sort_values("event_time", ascending=False).head(50),
                         use_container_width=True)

dashboard()

Writing app.py


In [61]:
import subprocess, time

subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.address", "0.0.0.0",
     "--server.headless", "true",
     "--server.enableCORS", "false",
     "--server.enableXsrfProtection", "false"],
    stdout=open("/tmp/st.log", "w"),
    stderr=subprocess.STDOUT
)
time.sleep(5)
print("done")

[16:44:59] Active streams: ['courier_windowed_to_blob(✅)', 'cancellations_to_blob(✅)', 'orders_to_blob(✅)', 'orders_windowed_to_blob(✅)', 'courier_to_blob(✅)', 'avg_order_value_to_blob(✅)', 'zone_demand_to_blob(✅)'] | Input rows: [7, 10735, 10652, 770, 1356, 0, 0]
done


### 13.4 Link to the dashboard

In [62]:
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
print(f"Open in new tab:\n{url}")

Open in new tab:
https://8501-m-s-kkb-usc1a0-1infd2g3gqfut-a.us-central1-0.prod.colab.dev
